# 01 · Public benchmarks: LFW and TinyFace

Compares the four embedders (ArcFace ResNet-50, ArcFace MobileFaceNet, SFace, LBP) on two public datasets. These are the numbers for Table IV of the paper.

- **LFW**: 6,000 face pairs; the model decides whether each pair shows the same person. Large, clear faces, so this is a sanity check.
- **TinyFace**: faces about 20×16 pixels, the size of faces at the back of a classroom. Tested with the official protocol, and with our open-set protocol where half the people are enrolled and everyone else must be rejected as unknown.

**Before running:**
1. Runtime → Change runtime type → **T4 GPU**.
2. Upload `quickpic-src.zip` (from `make bundle`) to `MyDrive/QuickPic/`.

Then Runtime → Run all. It takes about 30–45 minutes. Embeddings are cached in Drive, so a rerun after a disconnect resumes where it stopped. The last cell prints a summary to paste back.

No class photos are used here; only public datasets.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
QP = "/content/drive/MyDrive/QuickPic"

## 1. Install the code and GPU runtime

In [ ]:
!rm -rf /content/quickpic && mkdir -p /content/quickpic
!unzip -q -o "$QP/quickpic-src.zip" -d /content/quickpic
%cd /content/quickpic
!pip install -q -e . insightface scikit-image pillow-heif gdown
!pip uninstall -y -q onnxruntime && pip install -q --force-reinstall --no-deps onnxruntime-gpu

In [ ]:
import onnxruntime

providers = onnxruntime.get_available_providers()
print(providers)
assert "CUDAExecutionProvider" in providers, (
    "No GPU: Runtime > Change runtime type > T4 GPU, then rerun."
)

## 2. Download model weights and datasets

In [ ]:
!bash scripts/download_models.sh
!mkdir -p /content/lfw
![ -d /content/lfw/lfw ] || (curl -sL -o /content/lfw.tgz https://ndownloader.figshare.com/files/5976018 && tar -xzf /content/lfw.tgz -C /content/lfw)
![ -f /content/lfw/pairs.txt ] || curl -sL -o /content/lfw/pairs.txt https://ndownloader.figshare.com/files/5976006
![ -d /content/tinyface ] || (gdown -q 1xTZc7lNmWN33ECO2AKH6FycGdiqIK7W0 -O /content/tinyface.zip && unzip -q /content/tinyface.zip -d /content)
!ls /content/lfw/lfw | wc -l; ls /content/tinyface/Testing_Set/Gallery_Distractor | wc -l

## 3. Run the benchmarks

One embedder at a time, so a disconnect loses at most one. Each line prints its progress in images per second.

In [ ]:
import os

os.environ["QUICKPIC_DEVICE"] = "cuda"
for name in ["arcface_mbf", "arcface_r50", "sface", "lbp"]:
    !python -m experiments.public_benchmarks --lfw /content/lfw --tinyface /content/tinyface --cache "$QP/cache" --embedders {name}

## 4. Save and summarise

In [ ]:
import glob, json, shutil

os.makedirs(f"{QP}/results", exist_ok=True)
summary = {}
for path in sorted(glob.glob("experiments/results/public_benchmarks_*.json")):
    shutil.copy(path, f"{QP}/results/")
    record = json.load(open(path))
    r = record["results"]
    summary[r["embedder"]] = {
        "commit": record["commit"],
        "lfw_acc": round(r["lfw"]["accuracy"], 4),
        "lfw_tar@1e-3": round(r["lfw"]["tar_at_far_1e-3"], 4),
        "tf_rank1": round(r["tinyface"]["closed_set"]["rank1"], 4),
        "tf_rank20": round(r["tinyface"]["closed_set"]["rank20"], 4),
        "tf_distractors": r["tinyface"]["closed_set"]["distractor_images"],
        "tf_dir@1%": r["tinyface"]["open_set"]["dir_at_far_0.01"],
        "tf_dir@0.1%": r["tinyface"]["open_set"]["dir_at_far_0.001"],
    }
print("Paste everything below back to Claude:")
print(json.dumps(summary, indent=1))